# Project: Autodiff from scratch (a tiny PyTorch) ⚙️
`loss.backward()` is the most important line in deep learning. Build the engine behind it: a `Value` class that records every operation in a graph, then walks the graph backwards applying the chain rule (reverse-mode automatic differentiation, as in Karpathy's *micrograd*). Then train a small neural network with it.

Topic: [[Calculus and Optimization]], [[Backpropagation]], [[Matrix Calculus Cheatsheet]] · guide note: [[Project - Autodiff from Scratch]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import math, random, numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
class Value:
    def __init__(self, data, parents=()):
        self.data, self.grad, self._parents, self._backward = float(data), 0.0, parents, lambda: None
    def __repr__(self): return f"Value({self.data:.4f}, grad={self.grad:.4f})"
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other); out = Value(self.data + other.data, (self, other))
        def _backward(): self.grad += out.grad; other.grad += out.grad
        out._backward = _backward; return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # TODO 1: forward value, and a _backward that adds ∂out/∂self · out.grad and ∂out/∂other · out.grad
        raise NotImplementedError  # TODO 1
    def tanh(self):
        t = math.tanh(self.data); out = Value(t, (self,))
        def _backward(): self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward; return out
    def exp(self):
        out = Value(math.exp(self.data), (self,))
        def _backward(): self.grad += out.data * out.grad
        out._backward = _backward; return out
    def __pow__(self, k):
        out = Value(self.data ** k, (self,))
        def _backward(): self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward; return out
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other if isinstance(other, Value) else Value(-other))
    def __truediv__(self, other): return self * (other if isinstance(other, Value) else Value(other)) ** -1
    __radd__ = __add__
    def __rmul__(self, other): return self * other
    def backward(self):
        # TODO 2: topological order of the graph (parents before children), seed self.grad = 1, call _backward in reverse order
        raise NotImplementedError  # TODO 2

**TODO 1** (`__mul__`): for $z = xy$, $\partial z/\partial x = y$ and $\partial z/\partial y = x$. Use `+=`: a value used twice must collect gradient from both uses.

**TODO 2** (`backward`): a node's `_backward` may only run once its own `grad` is complete, i.e. after all nodes that use it. Build a topological order with a DFS, set the output's grad to 1 and run `_backward` from the output back to the inputs.

In [ ]:
def _t_ad():
    x, y = Value(2.0), Value(-3.0)
    z = x * y + x * x          # dz/dx = y + 2x = 1, dz/dy = x = 2
    z.backward(); ok1 = abs(x.grad - 1) < 1e-12 and abs(y.grad - 2) < 1e-12
    a = Value(0.5); f = (a * a * 3 + a.exp()).tanh(); f.backward()
    h = 1e-6; num = (math.tanh(3 * (0.5 + h) ** 2 + math.exp(0.5 + h)) - math.tanh(3 * (0.5 - h) ** 2 + math.exp(0.5 - h))) / (2 * h)
    return ok1 and abs(a.grad - num) < 1e-6
check("autodiff", _t_ad, "z = xy + x²: dz/dx = 1, dz/dy = 2; a composite function must match finite differences.")

In [ ]:
class Neuron:
    def __init__(self, n): self.w = [Value(random.uniform(-1, 1)) for _ in range(n)]; self.b = Value(0.0)
    def __call__(self, x): return sum((wi * xi for wi, xi in zip(self.w, x)), self.b).tanh()
    def params(self): return self.w + [self.b]
class MLP:
    def __init__(self, sizes): self.layers = [[Neuron(sizes[i]) for _ in range(sizes[i + 1])] for i in range(len(sizes) - 1)]
    def __call__(self, x):
        for layer in self.layers: x = [n(x) for n in layer]
        return x[0] if len(x) == 1 else x
    def params(self): return [p for layer in self.layers for n in layer for p in n.params()]

if ready("autodiff"):
    random.seed(0)
    from sklearn.datasets import make_moons
    X, y = make_moons(80, noise=0.1, random_state=0); y = y * 2 - 1                  # labels ±1
    model = MLP([2, 8, 8, 1]); hist = []
    for step in range(60):
        preds = [model(list(x)) for x in X]
        loss = sum(((p - yi) ** 2 for p, yi in zip(preds, y)), Value(0.0)) / len(X)
        for p in model.params(): p.grad = 0.0
        loss.backward()
        for p in model.params(): p.data -= 0.2 * p.grad
        hist.append(loss.data)
    acc = np.mean([(model(list(x)).data > 0) == (yi > 0) for x, yi in zip(X, y)])
    print(f"{len(model.params())} parameters, final loss {hist[-1]:.3f}, training accuracy {acc:.0%}")
    plt.plot(hist); plt.xlabel("step"); plt.ylabel("MSE"); plt.title("a neural net trained with YOUR autograd"); plt.show()
    check("trains a network", lambda: acc >= 0.9, "Your engine should train the MLP to ≥ 90 % accuracy on the moons.")

<details><summary>▶ Solution</summary>

```python
    def __mul__(self, other):   # after converting other to a Value
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
```

```python
    def backward(self):
        order, seen = [], set()
        def visit(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents: visit(p)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()
```
</details>

## Stretch goals
- Add `relu` and `log`, then train with cross-entropy instead of MSE.
- Check your gradients against PyTorch on the same expression (`torch.tensor(..., requires_grad=True)`).
- Make it vectorised: a `Tensor` class wrapping NumPy arrays (that's how PyTorch is organised).